In [1]:
import os
import json
import numpy as np
 
TSV_PATH        = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/data/raw/uniprotkb_AND_reviewed_true_AND_protein_2025_12_27.tsv"
FASTA_PATH      = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/data/raw/uniprotkb_AND_reviewed_true_AND_protein_2025_12_27.fasta"
GO_DICT_PATH    = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/data/processed/go_dict.json"
ASSIGNMENT_PATH = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/data/processed/go_group_assignment_v2.json"
PROCESSED_DIR   = "C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/data/processed/processed_data"
 
GO_COL_IDX    = 5    # zero-based column index for GO terms in TSV
ACCESSION_COL = 0    # zero-based column index for protein accession in TSV
 
# The groups that passed the size check and are ready to process first.
# We'll add sub-groups to this list later after the splitting cells run.
TRAIN_GROUPS = [
    "reproductive_process",
    "interspecies_interaction",
    "immune_system_process",
    "molecular_transducer",
    "mf_regulator_activity",
    "homeostatic_process",
    "atp_dependent_activity",
]
 
os.makedirs(PROCESSED_DIR, exist_ok=True)
print("Config ready")
print(f"  Output directory  : {PROCESSED_DIR}/")
print(f"  Groups to process : {len(TRAIN_GROUPS)}")

Config ready
  Output directory  : C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/data/processed/processed_data/
  Groups to process : 7


In [2]:
# Loads the parsed GO ontology and pre-computes the full ancestor set for every term.
# The ancestor cache is used for label propagation and group membership checks.
# This cell takes ~10-30 seconds — the recursion is the bottleneck.


def load_go_dict(path):
    with open(path, "r") as f:
        return json.load(f)


# Note: this uses recursion. sys.setrecursionlimit is set to 100000
# to handle deep GO hierarchies. data_pipeline.py uses an equivalent
# iterative version — both produce identical ancestor sets.
def build_ancestor_cache(go_dict):
    """
    Recursively compute the full ancestor set for every GO term.
    Uses an internal cache so each term is only traversed once.
    Ancestors = every term above this one in the GO hierarchy,
    not just direct parents.
    """
    cache = {}
 
    def get_ancestors(go_id):
        if go_id in cache:
            return cache[go_id]   # already done this one
        ancestors = set()
        term = go_dict.get(go_id)
        if term:
            for parent in term.get("parents", []):
                ancestors.add(parent)
                ancestors.update(get_ancestors(parent))   # climb higher
        cache[go_id] = ancestors
        return ancestors
 
    for go_id in go_dict:
        get_ancestors(go_id)
 
    return cache
 
go_dict        = load_go_dict(GO_DICT_PATH)
ancestor_cache = build_ancestor_cache(go_dict)
 
print(f"Loaded {len(go_dict):,} active GO terms from go_dict.json")
print(f"Ancestor cache built for {len(ancestor_cache):,} terms")

Loaded 38,560 active GO terms from go_dict.json
Ancestor cache built for 38,560 terms


In [4]:
# Loads the go_id → group mapping produced by go_group_assigner_v2.py.
# We pull out the "groups" key which maps each group name to its list of GO term IDs.
 
with open(ASSIGNMENT_PATH, "r") as f:
    assignment_data = json.load(f)
 
group_assignments = assignment_data["groups"]   # { group_name: [go_id, ...] }
 
print("Group assignments loaded:")
for group_name in TRAIN_GROUPS:
    n = len(group_assignments.get(group_name, []))
    flag = "" if n > 0 else "  <-- NOT FOUND, check assigner output"
    print(f"  {group_name:<35} {n:>4} GO terms{flag}")

Group assignments loaded:
  reproductive_process                 452 GO terms
  interspecies_interaction             424 GO terms
  immune_system_process                356 GO terms
  molecular_transducer                 320 GO terms
  mf_regulator_activity                308 GO terms
  homeostatic_process                  276 GO terms
  atp_dependent_activity               173 GO terms


In [5]:
# Reads the UniProtKB TSV and builds a map of accession → set of GO term IDs.
# We filter out any GO IDs that aren't in go_dict (obsolete or unresolved alt IDs)
# right here so downstream code never sees them.
 
def parse_tsv(tsv_path, go_col_idx, accession_col):
    protein_go  = {}
    rows_read   = 0
    rows_skipped = 0
 
    with open(tsv_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i == 0 or line.startswith("#") or not line.strip():
                continue   # skip header and blank lines
 
            parts = line.strip().split("\t")
            if len(parts) <= max(go_col_idx, accession_col):
                rows_skipped += 1
                continue   # row doesn't reach the columns we need
 
            accession = parts[accession_col].strip()
            raw_go    = parts[go_col_idx].strip()
 
            if not accession or not raw_go:
                continue
 
            go_ids = {g.strip() for g in raw_go.split(";") if g.strip()}
 
            # Filter to only GO IDs we actually know about — tosses obsolete IDs
            go_ids = go_ids & go_dict.keys()
 
            if go_ids:
                protein_go[accession] = go_ids
                rows_read += 1
 
    print(f"Parsed TSV: {rows_read:,} proteins with valid GO annotations")
    if rows_skipped > 0:
        print(f"Skipped {rows_skipped:,} rows that didn't reach the expected columns")
    if len(protein_go) < 1000:
        print("WARNING: very few proteins loaded — double-check GO_COL_IDX and ACCESSION_COL")
    return protein_go
 
protein_go = parse_tsv(TSV_PATH, GO_COL_IDX, ACCESSION_COL)
print(f"Loaded {len(protein_go):,} proteins from TSV")

Parsed TSV: 105,399 proteins with valid GO annotations
Skipped 527 rows that didn't reach the expected columns
Loaded 105,399 proteins from TSV


In [6]:
# Reads the FASTA and extracts accession → amino acid sequence.
# UniProtKB FASTA headers look like: >sp|P12345|GENE_HUMAN Description...
# We pull the accession from the second pipe-delimited field.
# After loading, we check how many proteins have both a sequence AND GO annotations.
 
def parse_fasta(fasta_path):
    sequences   = {}
    current_id  = None
    current_seq = []
 
    with open(fasta_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
 
            if line.startswith(">"):
                # Flush the sequence we just finished
                if current_id is not None:
                    sequences[current_id] = "".join(current_seq)
 
                header = line[1:]
                parts  = header.split("|")
                if len(parts) >= 2:
                    current_id = parts[1].strip()       # standard UniProtKB: sp|ACCESSION|...
                else:
                    current_id = parts[0].split()[0].strip()   # fallback for non-standard headers
                current_seq = []
            else:
                current_seq.append(line.upper())   # normalize to uppercase
 
    # The loop ends without a final ">" to trigger the last flush — do it manually
    if current_id is not None:
        sequences[current_id] = "".join(current_seq)
 
    return sequences
 
sequences = parse_fasta(FASTA_PATH)
print(f"Loaded {len(sequences):,} sequences from FASTA")
 
# How many proteins have both GO annotations and a sequence?
# If this is much lower than len(protein_go), TSV and FASTA might be from different releases.
overlap = set(protein_go.keys()) & set(sequences.keys())
print(f"Proteins with both sequence and GO annotations: {len(overlap):,}")
if len(overlap) / len(protein_go) < 0.8:
    print("WARNING: less than 80% overlap — check that TSV and FASTA are from the same Swiss-Prot release")

Loaded 105,951 sequences from FASTA
Proteins with both sequence and GO annotations: 105,399


In [7]:
# This function is the core of the True Path Rule implementation.
# Swiss-Prot only stores the most specific annotation per protein.
# Propagation adds all ancestor terms that are biologically implied.
#
# Example:
#   Input  : {GO:0010043}   <- zinc ion transport (most specific annotation stored)
#   Output : {GO:0010043, GO:0030001, GO:0006811, GO:0006810, GO:0051179, ...}
#             <- zinc ion transport + all ancestor terms up the hierarchy
#
# Without this, the model gets penalized for predicting correct ancestor terms
# that aren't in the raw label — a training error that silently tanks F1 scores.
 
def propagate_labels(go_ids, go_dict, ancestor_cache):
    """
    Expand a protein's GO annotations to include all ancestor terms.
    Filters result to only terms in go_dict (removes obsolete ancestors).
    """
    propagated = set(go_ids)
    for go_id in go_ids:
        propagated.update(ancestor_cache.get(go_id, set()))
    return propagated & go_dict.keys()   # drop anything we don't have a record for
 
print("Label propagation function defined and ready")

Label propagation function defined and ready


In [8]:
# For each of the 7 training groups:
#   1. Find proteins with at least one GO term in the group
#   2. Propagate labels (True Path Rule — fills in ancestor terms)
#   3. Build a binary label matrix (proteins × GO terms)
#   4. Apply frequency filter: drop terms appearing in < MIN_TERM_FREQ proteins
#      (terms that rare can't be learned reliably — they just add noise)
#   5. Save accessions, labels, go_terms, sequences, metadata to disk
#
# Output per group (inside processed_data/{group_name}/):
#   accessions.json  — list of protein accessions (row i = protein i in labels)
#   labels.npy       — binary matrix, shape (n_proteins, n_terms)
#   go_terms.json    — ordered list of GO term IDs (col j = term j in labels)
#   sequences.json   — {accession: sequence} for this group only
#   metadata.json    — stats summary
 
def build_and_save_group(
    group_name,
    group_go_terms,
    protein_go,
    sequences,
    go_dict,
    ancestor_cache,
    output_dir,
):
    group_go_set = set(group_go_terms)
    go_term_list = sorted(group_go_terms)     # alphabetical = reproducible column order
    term_to_idx  = {t: i for i, t in enumerate(go_term_list)}
    n_terms      = len(go_term_list)
 
    accessions    = []
    label_rows    = []
    group_seqs    = {}
    skipped_noseq = 0   # proteins in TSV but missing from FASTA
 
    for accession, raw_go_ids in protein_go.items():
        # Quick pre-filter before the expensive propagation call
        if not (raw_go_ids & group_go_set):
            continue   # this protein has no terms in this group at all
 
        if accession not in sequences:
            skipped_noseq += 1
            continue   # no sequence = can't extract features = can't train on it
 
        # Expand annotations to include all ancestor terms (True Path Rule)
        propagated = propagate_labels(raw_go_ids, go_dict, ancestor_cache)
 
        # Only care about terms that belong to this group
        relevant = propagated & group_go_set
        if not relevant:
            continue   # propagation didn't add anything useful for this group
 
        # Build the binary label vector for this protein
        label = np.zeros(n_terms, dtype=np.float32)
        for term in relevant:
            label[term_to_idx[term]] = 1.0
 
        accessions.append(accession)
        label_rows.append(label)
        group_seqs[accession] = sequences[accession]
 
    if not accessions:
        print(f"  {group_name}: No proteins found — skipping entirely")
        return None
 
    label_matrix = np.array(label_rows, dtype=np.float32)
 
    # Frequency filter — drop GO terms that appear too rarely to learn from.
    # MIN_TERM_FREQ=25 means: if a term is positive for fewer than 25 proteins,
    # we drop it. The model simply doesn't have enough examples to generalize.
    MIN_TERM_FREQ = 25
    term_freq     = label_matrix.sum(axis=0)
    keep_mask     = term_freq >= MIN_TERM_FREQ
    n_removed     = int((~keep_mask).sum())
    label_matrix  = label_matrix[:, keep_mask]
    go_term_list  = [t for t, k in zip(go_term_list, keep_mask) if k]
    n_terms       = len(go_term_list)
    print(f"      Frequency filter: kept {n_terms} terms, dropped {n_removed} rare terms (< {MIN_TERM_FREQ} proteins)")
 
    n_proteins    = len(accessions)
    label_density = label_matrix.mean()
    avg_terms     = label_density * n_terms
 
    # Create group output folder
    group_dir = os.path.join(output_dir, group_name)
    os.makedirs(group_dir, exist_ok=True)
 
    # Save accessions — row i in labels.npy = accessions[i]
    with open(os.path.join(group_dir, "accessions.json"), "w") as f:
        json.dump(accessions, f)
 
    # Save label matrix as numpy binary (fast to load)
    np.save(os.path.join(group_dir, "labels.npy"), label_matrix)
 
    # Save GO term order — column j in labels.npy = go_terms[j]
    # This mapping is critical — don't modify this file after training starts
    with open(os.path.join(group_dir, "go_terms.json"), "w") as f:
        json.dump(go_term_list, f, indent=2)
 
    # Save only the sequences for proteins in this group (keeps file sizes manageable)
    with open(os.path.join(group_dir, "sequences.json"), "w") as f:
        json.dump(group_seqs, f)
 
    # Save a human-readable stats summary
    with open(os.path.join(group_dir, "metadata.json"), "w") as f:
        json.dump({
            "group_name":            group_name,
            "n_proteins":            n_proteins,
            "n_go_terms":            n_terms,
            "label_density":         round(float(label_density), 6),
            "avg_terms_per_protein": round(float(avg_terms), 2),
            "skipped_no_seq":        skipped_noseq,
            "propagation":           "True Path Rule applied — ancestor terms included",
        }, f, indent=2)
 
    print(f"  {group_name}")
    print(f"      Proteins          : {n_proteins:,}")
    print(f"      GO terms          : {n_terms:,}")
    print(f"      Avg terms/protein : {avg_terms:.2f}")
    print(f"      Skipped (no seq)  : {skipped_noseq:,}")
    print(f"      Saved to          : {group_dir}/")
 
    return {"group_name": group_name, "n_proteins": n_proteins, "n_go_terms": n_terms,
            "label_density": round(float(label_density), 6),
            "avg_terms_per_protein": round(float(avg_terms), 2),
            "skipped_no_seq": skipped_noseq}
 
print("Building group datasets with annotation propagation...")
 
all_metadata = {}
 
for group_name in TRAIN_GROUPS:
    if group_name not in group_assignments:
        print(f"  '{group_name}' not in assignment file — skipping")
        continue
 
    meta = build_and_save_group(
        group_name     = group_name,
        group_go_terms = group_assignments[group_name],
        protein_go     = protein_go,
        sequences      = sequences,
        go_dict        = go_dict,
        ancestor_cache = ancestor_cache,
        output_dir     = PROCESSED_DIR,
    )
    if meta:
        all_metadata[group_name] = meta
 
# Save a combined summary so we can look back at all groups at once
summary_path = os.path.join(PROCESSED_DIR, "processing_summary.json")
with open(summary_path, "w") as f:
    json.dump(all_metadata, f, indent=2)
 
print(f"All {len(all_metadata)} groups processed")
print(f"Summary saved to {summary_path}")

Building group datasets with annotation propagation...
      Frequency filter: kept 129 terms, dropped 323 rare terms (< 25 proteins)
  reproductive_process
      Proteins          : 6,483
      GO terms          : 129
      Avg terms/protein : 3.53
      Skipped (no seq)  : 0
      Saved to          : C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/data/processed/processed_data\reproductive_process/
      Frequency filter: kept 168 terms, dropped 256 rare terms (< 25 proteins)
  interspecies_interaction
      Proteins          : 9,073
      GO terms          : 168
      Avg terms/protein : 5.78
      Skipped (no seq)  : 0
      Saved to          : C:/Users/USER/Documents/cod3astro/ML_AI/NeuralProt/data/processed/processed_data\interspecies_interaction/
      Frequency filter: kept 148 terms, dropped 208 rare terms (< 25 proteins)
  immune_system_process
      Proteins          : 5,079
      GO terms          : 148
      Avg terms/protein : 5.93
      Skipped (no seq)  : 0
      Sav

In [9]:
# Converts each protein sequence into a 428-dimensional feature vector.
# This only needs to run once per group — the output is saved as features.npy
# and loaded directly by the training script.
#
# Feature breakdown (428 total):
#   [0:20]    amino acid composition (20 features)
#             — fraction of each amino acid in the sequence
#   [20:420]  dipeptide composition (400 features)
#             — fraction of each possible pair of adjacent amino acids
#   [420:428] physicochemical properties (8 features)
#             — length, molecular weight, GRAVY score, aromaticity,
#               instability index, isoelectric point, charge at pH 7, aliphatic index
 
import itertools
 
# The 20 standard amino acids — our alphabet
AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")
 
# Molecular weight of each amino acid residue (Da)
AA_MW = {
    'A': 89.09,  'C': 121.16, 'D': 133.10, 'E': 147.13, 'F': 165.19,
    'G': 75.03,  'H': 155.16, 'I': 131.17, 'K': 146.19, 'L': 131.17,
    'M': 149.21, 'N': 132.12, 'P': 115.13, 'Q': 146.15, 'R': 174.20,
    'S': 105.09, 'T': 119.12, 'V': 117.15, 'W': 204.23, 'Y': 181.19,
}
 
# Kyte-Doolittle hydropathy values — positive = hydrophobic, negative = hydrophilic
AA_HYDRO = {
    'A':  1.8, 'C':  2.5, 'D': -3.5, 'E': -3.5, 'F':  2.8,
    'G': -0.4, 'H': -3.2, 'I':  4.5, 'K': -3.9, 'L':  3.8,
    'M':  1.9, 'N': -3.5, 'P': -1.6, 'Q': -3.5, 'R': -4.5,
    'S': -0.8, 'T': -0.7, 'V':  4.2, 'W': -0.9, 'Y': -1.3,
}
 
# Instability index dipeptide weights (Guruprasad et al., 1990)
# Only the "destabilizing" pairs have non-default weights — everything else is 1.0
INSTABILITY_WEIGHTS = {
    ('G','G'):13.34, ('G','D'):1.0,  ('G','E'):1.0,  ('G','H'):1.0,
    ('G','N'):1.0,   ('G','Q'):1.0,  ('G','R'):1.0,  ('G','S'):1.0,
    ('G','T'):1.0,   ('A','A'):1.0,  ('A','D'):1.0,  ('A','E'):1.0,
    ('A','H'):1.0,   ('A','K'):1.0,  ('A','N'):1.0,  ('A','Q'):1.0,
    ('A','R'):1.0,   ('A','S'):1.0,  ('A','T'):1.0,  ('C','K'):1.0,
    ('D','G'):1.0,   ('E','E'):1.0,  ('E','K'):1.0,  ('E','N'):1.0,
    ('E','Q'):1.0,   ('E','R'):1.0,  ('E','S'):1.0,  ('F','K'):1.0,
    ('H','H'):1.0,   ('H','K'):1.0,  ('H','N'):1.0,  ('H','Q'):1.0,
    ('H','R'):1.0,   ('H','S'):1.0,  ('I','F'):1.0,  ('I','K'):1.0,
    ('I','L'):1.0,   ('I','N'):1.0,  ('I','Q'):1.0,  ('I','R'):1.0,
    ('I','S'):1.0,   ('K','E'):1.0,  ('K','K'):1.0,  ('K','N'):1.0,
    ('K','P'):1.0,   ('K','Q'):1.0,  ('K','R'):1.0,  ('K','S'):1.0,
    ('L','F'):1.0,   ('L','K'):1.0,  ('L','N'):1.0,  ('L','Q'):1.0,
    ('L','R'):1.0,   ('L','S'):1.0,  ('M','K'):1.0,  ('N','D'):1.0,
    ('N','G'):1.0,   ('N','N'):1.0,  ('P','K'):1.0,  ('Q','D'):1.0,
    ('Q','E'):1.0,   ('Q','H'):1.0,  ('Q','K'):1.0,  ('Q','N'):1.0,
    ('Q','Q'):1.0,   ('Q','R'):1.0,  ('Q','S'):1.0,  ('R','H'):1.0,
    ('R','K'):1.0,   ('R','N'):1.0,  ('R','Q'):1.0,  ('R','R'):1.0,
    ('R','S'):1.0,   ('S','D'):1.0,  ('S','E'):1.0,  ('S','N'):1.0,
    ('S','S'):1.0,   ('T','K'):1.0,  ('V','K'):1.0,  ('W','K'):1.0,
    ('Y','K'):1.0,
}
 
# Pre-generate all 400 possible dipeptides (20 x 20) and their column indices
ALL_DIPEPTIDES = [''.join(p) for p in itertools.product(AA_LIST, repeat=2)]
DIPEPTIDE_IDX  = {dp: i for i, dp in enumerate(ALL_DIPEPTIDES)}
 
 
def compute_features(seq):
    """
    Compute 428 features for one protein sequence.
    Returns numpy array of shape (428,):
      [0:20]    amino acid composition
      [20:420]  dipeptide composition
      [420:428] physicochemical properties
    """
    # Normalize and strip non-standard characters
    seq = seq.upper()
    seq = ''.join(aa for aa in seq if aa in AA_LIST)
 
    if len(seq) == 0:
        return np.zeros(428, dtype=np.float32)   # empty sequence = all zeros
 
    n = len(seq)
 
    # Feature 1: Amino acid composition (20 features)
    # Just the fraction of each amino acid out of total sequence length
    aa_counts = {aa: 0 for aa in AA_LIST}
    for aa in seq:
        aa_counts[aa] += 1
    aac = np.array([aa_counts[aa] / n for aa in AA_LIST], dtype=np.float32)
 
    # Feature 2: Dipeptide composition (400 features)
    # Fraction of each adjacent amino acid pair out of all possible pairs
    dpc = np.zeros(400, dtype=np.float32)
    if n > 1:
        for i in range(n - 1):
            dp = seq[i] + seq[i + 1]
            if dp in DIPEPTIDE_IDX:
                dpc[DIPEPTIDE_IDX[dp]] += 1
        dpc /= (n - 1)   # normalize to fractions
 
    # Feature 3: Physicochemical properties (8 features)
 
    # 3a. Normalized length — log-scaled so huge proteins don't dominate
    norm_length = np.log1p(n) / np.log1p(35000)
 
    # 3b. Molecular weight — sum of residue weights minus water lost at each peptide bond
    mw      = sum(AA_MW.get(aa, 110.0) for aa in seq) - (n - 1) * 18.02
    norm_mw = mw / 1e6   # scale to ~[0, 1] range
 
    # 3c. GRAVY score — Grand Average of Hydropathy
    # Negative = hydrophilic (water-loving), positive = hydrophobic (membrane proteins tend here)
    gravy = sum(AA_HYDRO.get(aa, 0.0) for aa in seq) / n
 
    # 3d. Aromaticity — fraction of aromatic residues (Phe, Trp, Tyr)
    # Aromatic residues are involved in pi-stacking and binding interactions
    aromatic_count = sum(1 for aa in seq if aa in ('F', 'W', 'Y'))
    aromaticity    = aromatic_count / n
 
    # 3e. Instability index — predicts whether a protein is stable in solution
    # Values > 40 are considered unstable. We normalize to ~[0, 1].
    instability = 0.0
    if n > 1:
        for i in range(n - 1):
            instability += INSTABILITY_WEIGHTS.get((seq[i], seq[i+1]), 1.0)
        instability = (10.0 / n) * instability
    norm_instability = instability / 200.0
 
    # 3f. Isoelectric point (pI) — pH at which the protein carries no net charge
    # We find it via binary search on the charge function
    def charge_at_ph(s, ph):
        charge = 0.0
        # N-terminus and C-terminus contributions
        charge += 1.0 / (1.0 + 10 ** (ph - 8.0))
        charge -= 1.0 / (1.0 + 10 ** (3.1 - ph))
        # Side chain contributions per amino acid
        for aa in s:
            if   aa == 'K': charge += 1.0 / (1.0 + 10 ** (ph - 10.5))
            elif aa == 'R': charge += 1.0 / (1.0 + 10 ** (ph - 12.5))
            elif aa == 'H': charge += 1.0 / (1.0 + 10 ** (ph - 6.0))
            elif aa == 'D': charge -= 1.0 / (1.0 + 10 ** (3.9 - ph))
            elif aa == 'E': charge -= 1.0 / (1.0 + 10 ** (4.1 - ph))
            elif aa == 'C': charge -= 1.0 / (1.0 + 10 ** (8.3 - ph))
            elif aa == 'Y': charge -= 1.0 / (1.0 + 10 ** (10.1 - ph))
        return charge
 
    # Binary search for the pH where charge ≈ 0 (100 iterations = ~13 decimal places)
    lo, hi = 0.0, 14.0
    for _ in range(100):
        mid = (lo + hi) / 2.0
        if charge_at_ph(seq, mid) > 0:
            lo = mid
        else:
            hi = mid
    norm_pI = (lo + hi) / 2.0 / 14.0   # normalize to [0, 1]
 
    # 3g. Net charge at physiological pH (7.0) — clipped with tanh so large proteins
    # don't produce unbounded values
    norm_charge_ph7 = float(np.tanh(charge_at_ph(seq, 7.0) / 50.0))
 
    # 3h. Aliphatic index — relative volume occupied by aliphatic side chains
    # Higher = more thermostable. Formula from Ikai (1980).
    aliphatic = (
        aa_counts.get('A', 0) * 1.0 +
        aa_counts.get('V', 0) * 2.9 +
        aa_counts.get('I', 0) * 3.9 +
        aa_counts.get('L', 0) * 3.9
    ) / n * 100
    norm_aliphatic = aliphatic / 300.0
 
    physico = np.array([
        norm_length, norm_mw, gravy, aromaticity,
        norm_instability, norm_pI, norm_charge_ph7, norm_aliphatic,
    ], dtype=np.float32)
 
    return np.concatenate([aac, dpc, physico])
 
 
def compute_and_save_features(group_name, processed_dir):
    """
    Load a group's accessions and sequences, compute features for each protein,
    and save the result as features.npy in the group's folder.
    The training script loads this file — it never recomputes features itself.
    """
    group_dir = os.path.join(processed_dir, group_name)
    acc_path  = os.path.join(group_dir, "accessions.json")
    seq_path  = os.path.join(group_dir, "sequences.json")
    feat_path = os.path.join(group_dir, "features.npy")
 
    if not os.path.exists(seq_path):
        print(f"  {group_name}: sequences.json not found — run Cell 7 first")
        return
 
    with open(acc_path) as f: accessions = json.load(f)
    with open(seq_path) as f: group_seqs = json.load(f)
 
    n        = len(accessions)
    features = np.zeros((n, 428), dtype=np.float32)
 
    for i, acc in enumerate(accessions):
        features[i] = compute_features(group_seqs.get(acc, ""))
        if (i + 1) % 1000 == 0:
            print(f"    {i+1:,}/{n:,} sequences processed...")
 
    np.save(feat_path, features)
    print(f"  {group_name}  →  features.npy  shape: {features.shape}")
 
 
print("Computing features for all 7 groups — this is the slow step, grab a coffee...")
for group_name in TRAIN_GROUPS:
    compute_and_save_features(group_name, PROCESSED_DIR)
print("Feature extraction complete.")

Computing features for all 7 groups — this is the slow step, grab a coffee...
    1,000/6,483 sequences processed...
    2,000/6,483 sequences processed...
    3,000/6,483 sequences processed...
    4,000/6,483 sequences processed...
    5,000/6,483 sequences processed...
    6,000/6,483 sequences processed...
  reproductive_process  →  features.npy  shape: (6483, 428)
    1,000/9,073 sequences processed...
    2,000/9,073 sequences processed...
    3,000/9,073 sequences processed...
    4,000/9,073 sequences processed...
    5,000/9,073 sequences processed...
    6,000/9,073 sequences processed...
    7,000/9,073 sequences processed...
    8,000/9,073 sequences processed...
    9,000/9,073 sequences processed...
  interspecies_interaction  →  features.npy  shape: (9073, 428)
    1,000/5,079 sequences processed...
    2,000/5,079 sequences processed...
    3,000/5,079 sequences processed...
    4,000/5,079 sequences processed...
    5,000/5,079 sequences processed...
  immune_system_pr

In [10]:
compute_features("ACDEFGHIKLMNPQRSTVWY")

array([ 0.05      ,  0.05      ,  0.05      ,  0.05      ,  0.05      ,
        0.05      ,  0.05      ,  0.05      ,  0.05      ,  0.05      ,
        0.05      ,  0.05      ,  0.05      ,  0.05      ,  0.05      ,
        0.05      ,  0.05      ,  0.05      ,  0.05      ,  0.05      ,
        0.        ,  0.05263158,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.05263158,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.05263158,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.  

In [11]:
# Quick sanity check before handing off to training.
# Loads each group's files and confirms shapes are consistent.
# If anything is mismatched, re-run Cell 7 for that group.
 
print("Verifying saved data for all groups...")
 
all_ok = True
 
for group_name in TRAIN_GROUPS:
    group_dir = os.path.join(PROCESSED_DIR, group_name)
 
    required_files = ["accessions.json", "labels.npy", "go_terms.json",
                      "sequences.json", "metadata.json"]
 
    missing = [f for f in required_files
               if not os.path.exists(os.path.join(group_dir, f))]
 
    if missing:
        print(f"  {group_name} — MISSING FILES: {missing}")
        all_ok = False
        continue
 
    with open(os.path.join(group_dir, "accessions.json")) as f: acc   = json.load(f)
    with open(os.path.join(group_dir, "go_terms.json"))   as f: terms = json.load(f)
    with open(os.path.join(group_dir, "sequences.json"))  as f: seqs  = json.load(f)
    labels = np.load(os.path.join(group_dir, "labels.npy"))
 
    # Labels matrix should be (n_proteins, n_terms)
    shape_ok = (labels.shape == (len(acc), len(terms)))
    # sequences.json should have one entry per accession
    seqs_ok  = (len(seqs) == len(acc))
 
    ok = shape_ok and seqs_ok
    if not ok:
        all_ok = False
 
    status = "OK" if ok else "MISMATCH"
    print(f"  [{status}] {group_name}")
    print(f"         Proteins: {len(acc):,}  |  GO terms: {len(terms):,}  "
          f"|  Labels shape: {labels.shape}  |  Seqs: {len(seqs):,}")
    if not shape_ok:
        print(f"         Shape mismatch! Expected ({len(acc)}, {len(terms)}), got {labels.shape}")
    if not seqs_ok:
        print(f"         Sequence count mismatch! {len(seqs)} seqs vs {len(acc)} accessions")
 
if all_ok:
    print("\nAll groups verified")
else:
    print("\nSome groups failed verification — check errors above and re-run Cell 7 for those groups")

Verifying saved data for all groups...
  [OK] reproductive_process
         Proteins: 6,483  |  GO terms: 129  |  Labels shape: (6483, 129)  |  Seqs: 6,483
  [OK] interspecies_interaction
         Proteins: 9,073  |  GO terms: 168  |  Labels shape: (9073, 168)  |  Seqs: 9,073
  [OK] immune_system_process
         Proteins: 5,079  |  GO terms: 148  |  Labels shape: (5079, 148)  |  Seqs: 5,079
  [OK] molecular_transducer
         Proteins: 3,046  |  GO terms: 50  |  Labels shape: (3046, 50)  |  Seqs: 3,046
  [OK] mf_regulator_activity
         Proteins: 12,258  |  GO terms: 138  |  Labels shape: (12258, 138)  |  Seqs: 12,258
  [OK] homeostatic_process
         Proteins: 6,068  |  GO terms: 124  |  Labels shape: (6068, 124)  |  Seqs: 6,068
  [OK] atp_dependent_activity
         Proteins: 3,410  |  GO terms: 54  |  Labels shape: (3410, 54)  |  Seqs: 3,410

All groups verified


In [12]:
# After the first training run, 5 groups showed weak F1 scores (< 0.35).
# The fix: raise the frequency threshold from 25 → 50 to drop even rarer terms.
# Fewer but more learnable GO terms = better F1 in practice.
#
# After raising the threshold, some proteins end up with zero remaining positive labels
# (all their terms got filtered out). Those proteins are also removed — they carry
# no training signal and would just confuse the model.
 
WEAK_GROUPS = [
    "reproductive_process",
    "interspecies_interaction",
    "immune_system_process",
    "mf_regulator_activity",
    "homeostatic_process",
]
 
MIN_TERM_FREQ_NEW = 50   # raised from 25
 
print(f"Reprocessing {len(WEAK_GROUPS)} weak groups with MIN_TERM_FREQ = {MIN_TERM_FREQ_NEW}")
 
reprocess_metadata = {}
 
for group_name in WEAK_GROUPS:
    if group_name not in group_assignments:
        print(f"  '{group_name}' not found in assignment file — skipping")
        continue
 
    group_go_terms = group_assignments[group_name]
    group_go_set   = set(group_go_terms)
    go_term_list   = sorted(group_go_terms)
    term_to_idx    = {t: i for i, t in enumerate(go_term_list)}
    n_terms        = len(go_term_list)
 
    accessions    = []
    label_rows    = []
    group_seqs    = {}
    skipped_noseq = 0
 
    # Step 1: Build the full label matrix (same as Cell 7)
    for accession, raw_go_ids in protein_go.items():
        if not (raw_go_ids & group_go_set):
            continue
        if accession not in sequences:
            skipped_noseq += 1
            continue
 
        propagated = propagate_labels(raw_go_ids, go_dict, ancestor_cache)
        relevant   = propagated & group_go_set
        if not relevant:
            continue
 
        label = np.zeros(n_terms, dtype=np.float32)
        for term in relevant:
            label[term_to_idx[term]] = 1.0
 
        accessions.append(accession)
        label_rows.append(label)
        group_seqs[accession] = sequences[accession]
 
    if not accessions:
        print(f"  {group_name}: No proteins found — skipping")
        continue
 
    label_matrix = np.array(label_rows, dtype=np.float32)
 
    # Step 2: Apply the stricter frequency filter
    term_freq_old = label_matrix.sum(axis=0)
    n_terms_old   = n_terms
 
    keep_mask    = term_freq_old >= MIN_TERM_FREQ_NEW
    n_removed    = int((~keep_mask).sum())
    label_matrix = label_matrix[:, keep_mask]
    go_term_list = [t for t, k in zip(go_term_list, keep_mask) if k]
    n_terms      = len(go_term_list)
 
    # Step 3: Remove proteins that now have zero positive labels after the stricter filter
    # These proteins had all their terms filtered out — they're just dead weight now
    row_sums     = label_matrix.sum(axis=1)
    valid_mask   = row_sums > 0
    n_dropped    = int((~valid_mask).sum())
    label_matrix = label_matrix[valid_mask]
    accessions   = [a for a, v in zip(accessions, valid_mask) if v]
    group_seqs   = {a: group_seqs[a] for a in accessions}
 
    n_proteins    = len(accessions)
    label_density = label_matrix.mean()
    avg_terms     = label_density * n_terms
 
    # Step 4: Overwrite the old files with the updated data
    group_dir = os.path.join(PROCESSED_DIR, group_name)
    os.makedirs(group_dir, exist_ok=True)
 
    with open(os.path.join(group_dir, "accessions.json"), "w") as f: json.dump(accessions, f)
    np.save(os.path.join(group_dir, "labels.npy"), label_matrix)
    with open(os.path.join(group_dir, "go_terms.json"), "w") as f: json.dump(go_term_list, f, indent=2)
    with open(os.path.join(group_dir, "sequences.json"), "w") as f: json.dump(group_seqs, f)
    with open(os.path.join(group_dir, "metadata.json"), "w") as f:
        json.dump({
            "group_name":            group_name,
            "n_proteins":            n_proteins,
            "n_go_terms":            n_terms,
            "label_density":         round(float(label_density), 6),
            "avg_terms_per_protein": round(float(avg_terms), 2),
            "skipped_no_seq":        skipped_noseq,
            "dropped_zero_labels":   n_dropped,
            "min_term_freq":         MIN_TERM_FREQ_NEW,
            "propagation":           "True Path Rule applied — ancestor terms included",
        }, f, indent=2)
 
    reprocess_metadata[group_name] = {"n_proteins": n_proteins, "n_go_terms": n_terms}
 
    print(f"  {group_name}")
    print(f"      GO terms   : {n_terms_old} → {n_terms}  ({n_removed} rare terms removed)")
    print(f"      Proteins   : {n_proteins:,}  ({n_dropped} dropped — had zero labels after filter)")
    print(f"      Label density    : {label_density:.4f}")
    print(f"      Avg terms/protein: {avg_terms:.2f}")
 
print("Reprocessing complete")

Reprocessing 5 weak groups with MIN_TERM_FREQ = 50
  reproductive_process
      GO terms   : 452 → 72  (380 rare terms removed)
      Proteins   : 6,273  (210 dropped — had zero labels after filter)
      Label density    : 0.0464
      Avg terms/protein: 3.34
  interspecies_interaction
      GO terms   : 424 → 111  (313 rare terms removed)
      Proteins   : 8,924  (149 dropped — had zero labels after filter)
      Label density    : 0.0510
      Avg terms/protein: 5.66
  immune_system_process
      GO terms   : 356 → 106  (250 rare terms removed)
      Proteins   : 5,047  (32 dropped — had zero labels after filter)
      Label density    : 0.0536
      Avg terms/protein: 5.68
  mf_regulator_activity
      GO terms   : 308 → 98  (210 rare terms removed)
      Proteins   : 12,245  (13 dropped — had zero labels after filter)
      Label density    : 0.0527
      Avg terms/protein: 5.16
  homeostatic_process
      GO terms   : 276 → 88  (188 rare terms removed)
      Proteins   : 6,024  

In [13]:
# The protein list changed after reprocessing (some proteins were dropped),
# so we need to recompute features.npy with the updated accessions list.
# If we skip this, the features file will have the wrong number of rows.
 
print("Recomputing features for reprocessed weak groups...")
 
for group_name in WEAK_GROUPS:
    compute_and_save_features(group_name, PROCESSED_DIR)
 
print("Feature extraction complete for weak groups")

Recomputing features for reprocessed weak groups...
    1,000/6,273 sequences processed...
    2,000/6,273 sequences processed...
    3,000/6,273 sequences processed...
    4,000/6,273 sequences processed...
    5,000/6,273 sequences processed...
    6,000/6,273 sequences processed...
  reproductive_process  →  features.npy  shape: (6273, 428)
    1,000/8,924 sequences processed...
    2,000/8,924 sequences processed...
    3,000/8,924 sequences processed...
    4,000/8,924 sequences processed...
    5,000/8,924 sequences processed...
    6,000/8,924 sequences processed...
    7,000/8,924 sequences processed...
    8,000/8,924 sequences processed...
  interspecies_interaction  →  features.npy  shape: (8924, 428)
    1,000/5,047 sequences processed...
    2,000/5,047 sequences processed...
    3,000/5,047 sequences processed...
    4,000/5,047 sequences processed...
    5,000/5,047 sequences processed...
  immune_system_process  →  features.npy  shape: (5047, 428)
    1,000/12,245 seque

In [14]:
# Three large parent groups (catalytic_activity, localization, binding) have too
# many GO terms to train as single models — F1 drops badly above ~120 terms.
# We split them into biologically coherent sub-groups based on:
#   catalytic_activity → enzyme class (oxidoreductase, transferase, hydrolase, lyase)
#   localization       → transport mechanism + destination
#   binding            → molecular binding partner / chemical class
#
# Small sub-groups that don't have enough terms are merged into a related sub-group
# using MERGE_MAP (same tiebreaker logic as the main group assigner).
 
# Per-group frequency threshold overrides.
# Some sub-groups are dense enough that 25 is too low (too many terms pass the filter).
MIN_TERM_FREQ_SPLIT = 25    # default for most sub-groups
 
MIN_TERM_FREQ_OVERRIDES = {
    "transferase_activity": 50,   # 231 terms at threshold=25 — still too many
    "hydrolase_activity":   50,   # 232 terms at threshold=25 — still too many
    "protein_binding":      50,   # 259 terms at threshold=25 — still too many
}
 
# ── Catalytic Activity sub-groups (by Enzyme Commission class) ──
CATALYTIC_SUBGROUPS = {
    "oxidoreductase_activity": "GO:0016491",   # EC class 1 — oxidation/reduction reactions
    "transferase_activity":    "GO:0016740",   # EC class 2 (+ ligase merged in — both ATP-dependent)
    "hydrolase_activity":      "GO:0016787",   # EC class 3 — bond breaking with water
    "lyase_activity":          "GO:0016829",   # EC class 4 (+ isomerase merged in — both rearrange atoms)
}
 
CATALYTIC_MERGES = {
    "isomerase_activity": "lyase_activity",      # both rearrange atoms without adding/removing groups
    "ligase_activity":    "transferase_activity", # both involve ATP-dependent bond formation/transfer
}
 
CATALYTIC_MERGED_ROOTS = {
    "isomerase_activity": "GO:0016853",
    "ligase_activity":    "GO:0016874",
}
 
# ── Localization sub-groups (mechanism-based + destination-based) ──
# Original 5 sub-groups only captured ~28% of localization terms.
# Added destination-based sub-groups to capture the rest.
LOCALIZATION_SUBGROUPS = {
    # Mechanism-based (original set)
    "ion_transport":              "GO:0006811",
    "vesicle_mediated_transport": "GO:0016192",
    "protein_transport":          "GO:0015031",
    "lipid_transport":            "GO:0006869",
    # Destination-based (added to capture unmatched terms)
    "mitochondrial_targeting":    "GO:0006626",
    "er_golgi_transport":         "GO:0006888",
    "nuclear_transport":          "GO:0051169",
    "plasma_membrane_targeting":  "GO:0036010",
}
 
LOCALIZATION_MERGES = {
    "nucleocytoplasmic_transport": "nuclear_transport",   # most nucleocytoplasmic transport involves proteins
}
 
LOCALIZATION_MERGED_ROOTS = {
    "nucleocytoplasmic_transport": "GO:0006913",
}
 
# ── Binding sub-groups (by molecular partner / chemical class) ──
BINDING_SUBGROUPS = {
    # Molecular partner-based (original set)
    "protein_binding":        "GO:0005515",
    "dna_binding":            "GO:0003677",
    "rna_binding":            "GO:0003723",
    "lipid_binding":          "GO:0008289",
    # Chemical class-based (added to capture 367 previously unmatched terms)
    "metal_ion_binding":      "GO:0046872",   # zinc fingers, metalloenzymes, etc.
    "cofactor_binding":       "GO:0048037",   # vitamins, coenzymes
    "small_molecule_binding": "GO:0036094",   # drugs, metabolites, small ligands
}
 
BINDING_MERGES = {
    "carbohydrate_binding": "small_molecule_binding",   # carbohydrates are small molecules
}
 
BINDING_MERGED_ROOTS = {
    "carbohydrate_binding": "GO:0030246",
}
 
# Combined config — the splitting loop below iterates over this
SPLIT_CONFIG = {
    "catalytic_activity": {
        "subgroups":    CATALYTIC_SUBGROUPS,
        "merges":       CATALYTIC_MERGES,
        "merged_roots": CATALYTIC_MERGED_ROOTS,
    },
    "localization": {
        "subgroups":    LOCALIZATION_SUBGROUPS,
        "merges":       LOCALIZATION_MERGES,
        "merged_roots": LOCALIZATION_MERGED_ROOTS,
    },
    "binding": {
        "subgroups":    BINDING_SUBGROUPS,
        "merges":       BINDING_MERGES,
        "merged_roots": BINDING_MERGED_ROOTS,
    },
}
 
print("Sub-group splitting config loaded")
print(f"  catalytic_activity → {len(CATALYTIC_SUBGROUPS)} sub-groups (+ {len(CATALYTIC_MERGES)} merges)")
print(f"  localization       → {len(LOCALIZATION_SUBGROUPS)} sub-groups (+ {len(LOCALIZATION_MERGES)} merges)")
print(f"  binding            → {len(BINDING_SUBGROUPS)} sub-groups (+ {len(BINDING_MERGES)} merges)")
print(f"  Total new sub-groups: {len(CATALYTIC_SUBGROUPS)+len(LOCALIZATION_SUBGROUPS)+len(BINDING_SUBGROUPS)}")
print(f"\nMIN_TERM_FREQ overrides:")
for name, freq in MIN_TERM_FREQ_OVERRIDES.items():
    print(f"  {name}: {freq}")

Sub-group splitting config loaded
  catalytic_activity → 4 sub-groups (+ 2 merges)
  localization       → 8 sub-groups (+ 1 merges)
  binding            → 7 sub-groups (+ 1 merges)
  Total new sub-groups: 19

MIN_TERM_FREQ overrides:
  transferase_activity: 50
  hydrolase_activity: 50
  protein_binding: 50


In [15]:
# Before we run the splitting loop, check that every root GO ID in the config
# actually exists in go_dict. A typo in a GO ID will silently match nothing
# and produce empty sub-groups, which is hard to debug later.
 
print("Checking all sub-group root GO IDs against go_dict...")
all_valid = True
 
all_roots = {}
for parent, config in SPLIT_CONFIG.items():
    all_roots.update(config["subgroups"])
    all_roots.update(config["merged_roots"])
 
for name, root_id in sorted(all_roots.items()):
    if root_id in go_dict:
        term_name = go_dict[root_id].get("name", "unknown")
        print(f"  OK  {name:<40} {root_id}  ({term_name})")
    else:
        print(f"  MISSING  {name:<40} {root_id}  <-- not found in go_dict!")
        all_valid = False
 
if all_valid:
    print("\nAll root IDs valid — proceed to Cell 14")
else:
    print("\nFix the missing IDs above — they'll produce empty sub-groups")

Checking all sub-group root GO IDs against go_dict...
  OK  carbohydrate_binding                     GO:0030246  (carbohydrate binding)
  MISSING  cofactor_binding                         GO:0048037  <-- not found in go_dict!
  OK  dna_binding                              GO:0003677  (DNA binding)
  OK  er_golgi_transport                       GO:0006888  (endoplasmic reticulum to Golgi vesicle-mediated transport)
  OK  hydrolase_activity                       GO:0016787  (hydrolase activity)
  OK  ion_transport                            GO:0006811  (monoatomic ion transport)
  OK  isomerase_activity                       GO:0016853  (isomerase activity)
  OK  ligase_activity                          GO:0016874  (ligase activity)
  OK  lipid_binding                            GO:0008289  (lipid binding)
  OK  lipid_transport                          GO:0006869  (lipid transport)
  OK  lyase_activity                           GO:0016829  (lyase activity)
  OK  metal_ion_binding        

In [16]:
# Iterates over the three parent groups, assigns each of their GO terms to a
# sub-group using the same smallest-group tiebreaker as the main assigner,
# then builds and saves a dataset for each sub-group.
#
# The merge redirect works the same way as in go_group_assigner_v2 —
# merged group roots are included in the lookup so the tiebreaker can
# pick them, then they get redirected to their host sub-group.
 
def get_subgroup_sizes_all(go_dict, ancestor_cache, config):
    """Compute full ontology size for all sub-group roots including merged ones."""
    sizes = {}
    for parent, cfg in config.items():
        all_roots = {**cfg["subgroups"], **cfg["merged_roots"]}
        for name, root in all_roots.items():
            sizes[name] = sum(
                1 for go_id in go_dict
                if root in ancestor_cache.get(go_id, set())
            )
    return sizes
 
 
def assign_to_subgroup_v2(go_id, all_roots, ancestor_cache, subgroup_sizes):
    """
    Assign a single GO term to a sub-group (including merged group roots).
    Returns the sub-group name before merge redirection.
    """
    ancestors = ancestor_cache.get(go_id, set())
    matched   = [name for name, root in all_roots.items() if root in ancestors]
    if not matched:
        return None
    if len(matched) == 1:
        return matched[0]
    # Tiebreaker: pick the most specific (smallest) sub-group
    return min(matched, key=lambda n: subgroup_sizes.get(n, float("inf")))
 
 
def build_and_save_subgroup_v2(
    subgroup_name, subgroup_go_terms, protein_go, sequences,
    go_dict, ancestor_cache, output_dir, min_term_freq,
):
    """
    Build and save a dataset for one sub-group.
    Same logic as build_and_save_group in Cell 7, but with configurable
    min_term_freq and the extra step of dropping zero-label proteins after filtering.
    Returns (metadata_dict, None) on success, or (None, error_message) on failure.
    """
    group_go_set = set(subgroup_go_terms)
    go_term_list = sorted(subgroup_go_terms)
    term_to_idx  = {t: i for i, t in enumerate(go_term_list)}
    n_terms      = len(go_term_list)
 
    accessions = []
    label_rows = []
    group_seqs = {}
    skipped    = 0
 
    for accession, raw_go_ids in protein_go.items():
        if not (raw_go_ids & group_go_set):
            continue
        if accession not in sequences:
            skipped += 1
            continue
        propagated = propagate_labels(raw_go_ids, go_dict, ancestor_cache)
        relevant   = propagated & group_go_set
        if not relevant:
            continue
        label = np.zeros(n_terms, dtype=np.float32)
        for term in relevant:
            label[term_to_idx[term]] = 1.0
        accessions.append(accession)
        label_rows.append(label)
        group_seqs[accession] = sequences[accession]
 
    if not accessions:
        return None, "no proteins found"
 
    label_matrix = np.array(label_rows, dtype=np.float32)
 
    # Drop terms below the frequency threshold
    keep_mask    = label_matrix.sum(axis=0) >= min_term_freq
    n_removed    = int((~keep_mask).sum())
    label_matrix = label_matrix[:, keep_mask]
    go_term_list = [t for t, k in zip(go_term_list, keep_mask) if k]
    n_terms      = len(go_term_list)
 
    if n_terms == 0:
        return None, "all terms removed by frequency filter — lower min_term_freq"
 
    # Drop proteins that have zero positive labels after the frequency filter
    valid_mask   = label_matrix.sum(axis=1) > 0
    n_dropped    = int((~valid_mask).sum())
    label_matrix = label_matrix[valid_mask]
    accessions   = [a for a, v in zip(accessions, valid_mask) if v]
    group_seqs   = {a: group_seqs[a] for a in accessions}
 
    if len(accessions) < 50:
        return None, f"too few proteins after filtering ({len(accessions)} < 50)"
 
    n_proteins    = len(accessions)
    label_density = label_matrix.mean()
    avg_terms     = label_density * n_terms
 
    group_dir = os.path.join(output_dir, subgroup_name)
    os.makedirs(group_dir, exist_ok=True)
 
    with open(os.path.join(group_dir, "accessions.json"), "w") as f: json.dump(accessions, f)
    np.save(os.path.join(group_dir, "labels.npy"), label_matrix)
    with open(os.path.join(group_dir, "go_terms.json"), "w") as f: json.dump(go_term_list, f, indent=2)
    with open(os.path.join(group_dir, "sequences.json"), "w") as f: json.dump(group_seqs, f)
    with open(os.path.join(group_dir, "metadata.json"), "w") as f:
        json.dump({
            "group_name":            subgroup_name,
            "n_proteins":            n_proteins,
            "n_go_terms":            n_terms,
            "n_terms_removed":       n_removed,
            "n_proteins_dropped":    n_dropped,
            "label_density":         round(float(label_density), 6),
            "avg_terms_per_protein": round(float(avg_terms), 2),
            "min_term_freq":         min_term_freq,
            "propagation":           "True Path Rule applied",
        }, f, indent=2)
 
    return {
        "group_name": subgroup_name, "n_proteins": n_proteins, "n_go_terms": n_terms,
        "n_terms_removed": n_removed, "n_proteins_dropped": n_dropped,
        "label_density": round(float(label_density), 6),
        "avg_terms_per_protein": round(float(avg_terms), 2),
        "min_term_freq": min_term_freq,
    }, None
 
 
print("Computing sub-group sizes (needed for tiebreaker)...")
subgroup_sizes  = get_subgroup_sizes_all(go_dict, ancestor_cache, SPLIT_CONFIG)
new_group_names = []
 
for parent_group, config in SPLIT_CONFIG.items():
    print(f"\n{'─'*60}")
    print(f"Splitting: {parent_group}")
 
    if parent_group not in group_assignments:
        print(f"  Not found in group_assignments — did the assigner run successfully?")
        continue
 
    parent_go_terms = group_assignments[parent_group]
    subgroups       = config["subgroups"]
    merges          = config["merges"]
    merged_roots    = config["merged_roots"]
    all_roots       = {**subgroups, **merged_roots}
 
    print(f"  Parent has {len(parent_go_terms):,} GO terms to distribute")
 
    # Distribute each parent GO term into a sub-group
    subgroup_term_lists = {name: [] for name in subgroups}
    unmatched           = []
 
    for go_id in parent_go_terms:
        raw_assigned = assign_to_subgroup_v2(go_id, all_roots, ancestor_cache, subgroup_sizes)
        if raw_assigned is None:
            unmatched.append(go_id)
        else:
            final = merges.get(raw_assigned, raw_assigned)
            if final in subgroup_term_lists:
                subgroup_term_lists[final].append(go_id)
            else:
                unmatched.append(go_id)   # assigned to a merged group that has no host here
 
    print(f"  Unmatched terms: {len(unmatched)} (will not be trained on)")
    print(f"  Distribution before frequency filter:")
    for name, terms in sorted(subgroup_term_lists.items(), key=lambda x: -len(x[1])):
        print(f"    {name:<40} {len(terms):>5} terms")
 
    print(f"\n  Building sub-group datasets:")
    for subgroup_name, subgroup_terms in subgroup_term_lists.items():
        if not subgroup_terms:
            print(f"    {subgroup_name}: no GO terms — skipping")
            continue
 
        freq = MIN_TERM_FREQ_OVERRIDES.get(subgroup_name, MIN_TERM_FREQ_SPLIT)
 
        meta, error = build_and_save_subgroup_v2(
            subgroup_name     = subgroup_name,
            subgroup_go_terms = subgroup_terms,
            protein_go        = protein_go,
            sequences         = sequences,
            go_dict           = go_dict,
            ancestor_cache    = ancestor_cache,
            output_dir        = PROCESSED_DIR,
            min_term_freq     = freq,
        )
 
        if meta is None:
            print(f"    {subgroup_name}: SKIPPED — {error}")
            continue
 
        print(f"    {subgroup_name}")
        print(f"         Proteins: {meta['n_proteins']:,}  |  "
              f"GO terms: {meta['n_go_terms']}  |  "
              f"Freq filter: {freq}  |  "
              f"Avg terms/protein: {meta['avg_terms_per_protein']:.2f}")
 
        new_group_names.append(subgroup_name)
 
print(f"\n{len(new_group_names)} sub-groups created successfully")
print("Add these to TRAIN_GROUPS in train_model.ipynb:")
for name in new_group_names:
    print(f'    "{name}",')

Computing sub-group sizes (needed for tiebreaker)...

────────────────────────────────────────────────────────────
Splitting: catalytic_activity
  Parent has 4,846 GO terms to distribute
  Unmatched terms: 50 (will not be trained on)
  Distribution before frequency filter:
    transferase_activity                      1810 terms
    oxidoreductase_activity                   1338 terms
    hydrolase_activity                         996 terms
    lyase_activity                             652 terms

  Building sub-group datasets:
    oxidoreductase_activity
         Proteins: 8,673  |  GO terms: 159  |  Freq filter: 25  |  Avg terms/protein: 2.86
    transferase_activity
         Proteins: 15,919  |  GO terms: 154  |  Freq filter: 50  |  Avg terms/protein: 4.04
    hydrolase_activity
         Proteins: 15,152  |  GO terms: 131  |  Freq filter: 50  |  Avg terms/protein: 4.37
    lyase_activity
         Proteins: 4,756  |  GO terms: 75  |  Freq filter: 25  |  Avg terms/protein: 2.37

─────

In [17]:
# Run feature extraction for every sub-group created in Cell 14.
# The training script won't be able to load a group without features.npy.
 
print("Extracting features for new sub-groups...")
print("=" * 60)
 
for subgroup_name in new_group_names:
    group_dir = os.path.join(PROCESSED_DIR, subgroup_name)
    if not os.path.isdir(group_dir):
        print(f"  {subgroup_name}: folder not found — did Cell 14 skip this one?")
        continue
    compute_and_save_features(subgroup_name, PROCESSED_DIR)
 
print("Feature extraction complete for all new sub-groups")
print("The group names to paste into TRAIN_GROUPS in train_model.ipynb:")
for name in new_group_names:
    print(f'    "{name}",')

Extracting features for new sub-groups...
    1,000/8,673 sequences processed...
    2,000/8,673 sequences processed...
    3,000/8,673 sequences processed...
    4,000/8,673 sequences processed...
    5,000/8,673 sequences processed...
    6,000/8,673 sequences processed...
    7,000/8,673 sequences processed...
    8,000/8,673 sequences processed...
  oxidoreductase_activity  →  features.npy  shape: (8673, 428)
    1,000/15,919 sequences processed...
    2,000/15,919 sequences processed...
    3,000/15,919 sequences processed...
    4,000/15,919 sequences processed...
    5,000/15,919 sequences processed...
    6,000/15,919 sequences processed...
    7,000/15,919 sequences processed...
    8,000/15,919 sequences processed...
    9,000/15,919 sequences processed...
    10,000/15,919 sequences processed...
    11,000/15,919 sequences processed...
    12,000/15,919 sequences processed...
    13,000/15,919 sequences processed...
    14,000/15,919 sequences processed...
    15,000/15,919 

In [18]:
# After training, these two groups had the weakest results despite 100 epochs.
# protein_binding (GO:0005515) is an extremely broad category — proteins bind
# other proteins for wildly different reasons, and 259 terms was too many.
# Raising min_freq to 100/75 aggressively cuts the label space to only the
# most common and learnable terms.
 
REPROCESS_TARGETS = {
    "protein_binding":        {"source_group": "binding", "root": "GO:0005515", "min_freq": 100},
    "small_molecule_binding": {"source_group": "binding", "root": "GO:0036094", "min_freq": 75},
}
 
# Full binding sub-group config — needed to correctly re-assign GO terms
BINDING_SUBGROUPS_FULL = {
    "protein_binding":        "GO:0005515",
    "dna_binding":            "GO:0003677",
    "rna_binding":            "GO:0003723",
    "lipid_binding":          "GO:0008289",
    "metal_ion_binding":      "GO:0046872",
    "small_molecule_binding": "GO:0036094",
}
 
BINDING_MERGED_ROOTS_FULL = {
    "carbohydrate_binding": "GO:0030246",
}
 
BINDING_MERGES_FULL = {
    "carbohydrate_binding": "small_molecule_binding",
}
 
ALL_BINDING_ROOTS = {**BINDING_SUBGROUPS_FULL, **BINDING_MERGED_ROOTS_FULL}
 
print("Reprocessing protein_binding and small_molecule_binding with stricter frequency filter...")
print("=" * 60)
 
 
def get_subgroup_sizes_local(go_dict, ancestor_cache, roots):
    """Compute ontology sizes for a given set of roots — used as tiebreaker."""
    return {
        name: sum(1 for go_id in go_dict if root in ancestor_cache.get(go_id, set()))
        for name, root in roots.items()
    }
 
 
def assign_binding_term(go_id, ancestor_cache, all_roots, subgroup_sizes, merges):
    """Assign one binding GO term to a sub-group and apply merge if needed."""
    ancestors = ancestor_cache.get(go_id, set())
    matched   = [name for name, root in all_roots.items() if root in ancestors]
    if not matched:
        return None
    raw = matched[0] if len(matched) == 1 else min(matched, key=lambda n: subgroup_sizes.get(n, float("inf")))
    return merges.get(raw, raw)
 
 
if "binding" not in group_assignments:
    print("'binding' not found in group_assignments — make sure Cell 3 ran successfully")
else:
    parent_go_terms = group_assignments["binding"]
    subgroup_sizes  = get_subgroup_sizes_local(go_dict, ancestor_cache, ALL_BINDING_ROOTS)
 
    # Redistribute all binding GO terms into sub-groups
    subgroup_term_lists = {name: [] for name in BINDING_SUBGROUPS_FULL}
    for go_id in parent_go_terms:
        assigned = assign_binding_term(go_id, ancestor_cache, ALL_BINDING_ROOTS, subgroup_sizes, BINDING_MERGES_FULL)
        if assigned and assigned in subgroup_term_lists:
            subgroup_term_lists[assigned].append(go_id)
 
    reprocessed = []
 
    for group_name, config in REPROCESS_TARGETS.items():
        min_freq       = config["min_freq"]
        subgroup_terms = subgroup_term_lists.get(group_name, [])
 
        if not subgroup_terms:
            print(f"No GO terms found for {group_name} — something went wrong in the assignment above")
            continue
 
        print(f"\nReprocessing: {group_name}")
        print(f"  GO terms before filter : {len(subgroup_terms)}")
        print(f"  New MIN_TERM_FREQ       : {min_freq}")
 
        group_go_set = set(subgroup_terms)
        go_term_list = sorted(subgroup_terms)
        term_to_idx  = {t: i for i, t in enumerate(go_term_list)}
        n_terms      = len(go_term_list)
 
        accessions = []
        label_rows = []
        group_seqs = {}
        skipped    = 0
 
        # Rebuild the label matrix from scratch with propagation
        for accession, raw_go_ids in protein_go.items():
            if not (raw_go_ids & group_go_set):
                continue
            if accession not in sequences:
                skipped += 1
                continue
            propagated = propagate_labels(raw_go_ids, go_dict, ancestor_cache)
            relevant   = propagated & group_go_set
            if not relevant:
                continue
            label = np.zeros(n_terms, dtype=np.float32)
            for term in relevant:
                label[term_to_idx[term]] = 1.0
            accessions.append(accession)
            label_rows.append(label)
            group_seqs[accession] = sequences[accession]
 
        if not accessions:
            print(f"  No proteins found — skipping")
            continue
 
        label_matrix = np.array(label_rows, dtype=np.float32)
 
        # Apply the new, stricter frequency filter
        term_freq    = label_matrix.sum(axis=0)
        keep_mask    = term_freq >= min_freq
        n_removed    = int((~keep_mask).sum())
        label_matrix = label_matrix[:, keep_mask]
        go_term_list = [t for t, k in zip(go_term_list, keep_mask) if k]
        n_terms      = len(go_term_list)
 
        if n_terms == 0:
            print(f"  All terms removed — try lowering min_freq for this group")
            continue
 
        # Drop proteins with zero labels after the filter
        valid_mask   = label_matrix.sum(axis=1) > 0
        n_dropped    = int((~valid_mask).sum())
        label_matrix = label_matrix[valid_mask]
        accessions   = [a for a, v in zip(accessions, valid_mask) if v]
        group_seqs   = {a: group_seqs[a] for a in accessions}
 
        n_proteins    = len(accessions)
        label_density = label_matrix.mean()
        avg_terms     = label_density * n_terms
 
        # Overwrite the existing files for this group
        group_dir = os.path.join(PROCESSED_DIR, group_name)
        os.makedirs(group_dir, exist_ok=True)
 
        with open(os.path.join(group_dir, "accessions.json"), "w") as f: json.dump(accessions, f)
        np.save(os.path.join(group_dir, "labels.npy"), label_matrix)
        with open(os.path.join(group_dir, "go_terms.json"), "w") as f: json.dump(go_term_list, f, indent=2)
        with open(os.path.join(group_dir, "sequences.json"), "w") as f: json.dump(group_seqs, f)
        with open(os.path.join(group_dir, "metadata.json"), "w") as f:
            json.dump({
                "group_name":            group_name,
                "n_proteins":            n_proteins,
                "n_go_terms":            n_terms,
                "n_terms_removed":       n_removed,
                "n_proteins_dropped":    n_dropped,
                "label_density":         round(float(label_density), 6),
                "avg_terms_per_protein": round(float(avg_terms), 2),
                "skipped_no_seq":        skipped,
                "min_term_freq":         min_freq,
                "propagation":           "True Path Rule applied",
            }, f, indent=2)
 
        print(f"  Saved")
        print(f"      GO terms   : {len(subgroup_terms)} → {n_terms} ({n_removed} rare terms removed)")
        print(f"      Proteins   : {n_proteins:,} ({n_dropped} dropped — zero labels after filter)")
        print(f"      Label density    : {label_density:.4f}")
        print(f"      Avg terms/protein: {avg_terms:.2f}")
 
        reprocessed.append(group_name)
 
    print(f"\nReprocessing complete for: {reprocessed}")
 
    # Recompute features since the protein list changed
    print("\nRecomputing features for reprocessed groups...")
    print("=" * 60)
    for group_name in reprocessed:
        compute_and_save_features(group_name, PROCESSED_DIR)
 
    print("Feature extraction complete")
    print("Next: in train_model.ipynb Cell 2, set:")
    print("    EPOCHS = 100")
    print("    TRAIN_GROUPS = [")
    print('        "mf_regulator_activity",')
    print('        "protein_binding",')
    print('        "small_molecule_binding",')
    print("    ]")

Reprocessing protein_binding and small_molecule_binding with stricter frequency filter...

Reprocessing: protein_binding
  GO terms before filter : 667
  New MIN_TERM_FREQ       : 100
  Saved
      GO terms   : 667 → 97 (570 rare terms removed)
      Proteins   : 23,208 (533 dropped — zero labels after filter)
      Label density    : 0.0302
      Avg terms/protein: 2.93

Reprocessing: small_molecule_binding
  GO terms before filter : 191
  New MIN_TERM_FREQ       : 75
  Saved
      GO terms   : 191 → 57 (134 rare terms removed)
      Proteins   : 27,779 (71 dropped — zero labels after filter)
      Label density    : 0.1272
      Avg terms/protein: 7.25

Reprocessing complete for: ['protein_binding', 'small_molecule_binding']

Recomputing features for reprocessed groups...
    1,000/23,208 sequences processed...
    2,000/23,208 sequences processed...
    3,000/23,208 sequences processed...
    4,000/23,208 sequences processed...
    5,000/23,208 sequences processed...
    6,000/23,20